# Upload CATI Weights to HuggingFace Hub
Run all cells top to bottom once. Takes ~2 min depending on Drive speed.

In [ ]:
# ── Cell 1: Mount Drive & verify checkpoints exist ────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os

CATI_WEIGHTS = '/content/drive/MyDrive/sg_smart_city/models/phase2/cati_phase2_final.pt'
YOLO_WEIGHTS = '/content/drive/MyDrive/sg_smart_city/models/phase2/yolo_cati6/weights/best.pt'

for label, path in [('CATI module', CATI_WEIGHTS), ('YOLO backbone', YOLO_WEIGHTS)]:
    size_mb = os.path.getsize(path) / 1e6 if os.path.exists(path) else None
    status = f'{size_mb:.1f} MB ✅' if size_mb else '❌ NOT FOUND'
    print(f'{label}: {status}')

In [ ]:
# ── Cell 2: Install huggingface_hub if not already present ───────────────────
!pip install -q huggingface_hub

In [ ]:
# ── Cell 3: Upload both files to HF Hub ──────────────────────────────────────
# Add your HF token in Colab Secrets: click the key icon (🔑) in the left sidebar
# → New secret → Name: HF_TOKEN → Value: your token
from google.colab import userdata
from huggingface_hub import HfApi, create_repo

HF_TOKEN = userdata.get('HF_TOKEN')
REPO_ID  = 'SuhxsReddy/cati-singapore'

api = HfApi()
create_repo(REPO_ID, token=HF_TOKEN, repo_type='model', exist_ok=True)
print(f'Repo ready: https://huggingface.co/{REPO_ID}')

uploads = {
    'cati_best.pt':     CATI_WEIGHTS,   # Phase 2 CATI conditioning module
    'yolo_backbone.pt': YOLO_WEIGHTS,   # Phase 2 fine-tuned YOLO backbone
}

for hf_name, local_path in uploads.items():
    print(f'\nUploading {hf_name} ...')
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=hf_name,
        repo_id=REPO_ID,
        token=HF_TOKEN,
    )
    print(f'  ✅ https://huggingface.co/{REPO_ID}/blob/main/{hf_name}')

print('\nAll done — Space will load these on next deploy.')